### 1- EQL TRIANING (SP_GR DAYASET)

In [4]:
# ============================================================
# EQL SYMBOLIC REGRESSION — SP_GR
# CONTROLLED OPERATORS
#
# Operators intended:
#     +  -  *  /  log  exp  sqrt
#
# Same grouped frozen split as Paper 1 / PySR / AI Feynman
# ============================================================

import os
import random
import inspect
import re

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from EQL.model import EQL
from EQL.layer import EqlLayer


# ============================================================
# 1) REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2) FILES / DIRECTORIES
# ============================================================

DATA_FILE = "sp_gr_dataset_clean.csv"

RESULT_DIR = "results_EQL_SP_GR_Controlled"

os.makedirs(RESULT_DIR, exist_ok=True)


# ============================================================
# 3) LOAD DATA
# ============================================================

df = pd.read_csv(DATA_FILE)

df = df.rename(columns={
    "Specific Gravity": "SG",
    "Pressure (psia)": "P",
    "Hydrate Temperature (K)": "HFT"
})

df = df[["SG", "P", "HFT"]]

df = (
    df
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .reset_index(drop=True)
)


# ============================================================
# 4) INPUT / TARGET
# ============================================================

X = df[["SG", "P"]].copy()

y = df["HFT"].astype(float).values


# ============================================================
# 5) SAME GROUPING AS PAPER 1
# ============================================================

df["SG_round"] = df["SG"].round(3)

df["P_round"] = df["P"].round(3)

df["group_id"] = (
    df[["SG_round", "P_round"]]
    .astype(str)
    .agg("_".join, axis=1)
)

groups = df["group_id"].values


# ============================================================
# 6) FROZEN GROUPED 80/20 SPLIT
# ============================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    gss.split(X, y, groups)
)


X_train = X.iloc[train_idx].reset_index(drop=True)

X_test = X.iloc[test_idx].reset_index(drop=True)

y_train = y[train_idx]

y_test = y[test_idx]

groups_train = groups[train_idx]

groups_test = groups[test_idx]


# ============================================================
# 7) LEAKAGE CHECK
# ============================================================

overlap = set(groups_train) & set(groups_test)

assert len(overlap) == 0, (
    f"Data leakage detected: {len(overlap)} overlapping groups."
)


print("\n============================================================")
print("FROZEN GROUPED SPLIT")
print("============================================================")

print(f"Total observations : {len(df)}")
print(f"Training           : {len(X_train)}")
print(f"Testing            : {len(X_test)}")

print(f"Training groups    : {len(set(groups_train))}")
print(f"Testing groups     : {len(set(groups_test))}")
print(f"Group overlap      : {len(overlap)}")


# ============================================================
# 8) SAVE FROZEN INDICES
# ============================================================

pd.DataFrame({
    "train_idx": train_idx
}).to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_train_idx.csv"
    ),
    index=False
)


pd.DataFrame({
    "test_idx": test_idx
}).to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_test_idx.csv"
    ),
    index=False
)


# ============================================================
# 9) METRICS
# ============================================================

def calculate_metrics(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    rrmse = (
        100.0
        * rmse
        / np.mean(np.abs(y_true))
    )

    return {
        "RMSE_K": rmse,
        "MAE_K": mae,
        "RRMSE_%": rrmse,
        "R2": r2
    }


# ============================================================
# 10) STANDARDIZATION
#
# IMPORTANT:
# Mean/std are calculated from TRAINING DATA ONLY.
# ============================================================

X_mean = X_train.mean()

X_std = X_train.std()

X_train_scaled = (
    X_train - X_mean
) / X_std

X_test_scaled = (
    X_test - X_mean
) / X_std


X_train_np = X_train_scaled.values.astype(
    np.float32
)

X_test_np = X_test_scaled.values.astype(
    np.float32
)

y_train_np = y_train.astype(
    np.float32
)

y_test_np = y_test.astype(
    np.float32
)


# ============================================================
# 11) EQL OPERATOR INSPECTION
# ============================================================

print("\n============================================================")
print("EQL OPERATOR INFORMATION")
print("============================================================")

print(inspect.signature(EqlLayer))


# ============================================================
# 12) CREATE EQL MODEL
# ============================================================

model = EQL(
    num_layers=2,
    dim=2,
    v=[1, 1]
)


# ============================================================
# 13) BUILD / COMPILE
# ============================================================

model.build_and_compile_model()

print("\n============================================================")
print("EQL MODEL")
print("============================================================")

model.summary()


# ============================================================
# 14) TRAINING
#
# T0 = 2000 epochs
# T1 = 1000 epochs
# T2 = 1000 epochs
#
# verbose=0 hides epoch-by-epoch output.
# ============================================================

print("\n============================================================")
print("EQL TRAINING")
print("============================================================")

model.fit(
    X_train_np,
    y_train_np,

    lmbda=0.001,

    t0=2000,
    t1=1000,
    t2=1000,

    batch_size=32,

    shuffle=True,

    verbose=0
)

print("EQL training completed.")


# ============================================================
# 15) PREDICTIONS
# ============================================================

pred_train = np.asarray(
    model.predict(
        X_train_np,
        verbose=0
    )
).ravel()


pred_test = np.asarray(
    model.predict(
        X_test_np,
        verbose=0
    )
).ravel()


# ============================================================
# 16) METRICS
# ============================================================

train_metrics = calculate_metrics(
    y_train,
    pred_train
)

test_metrics = calculate_metrics(
    y_test,
    pred_test
)


print("\n============================================================")
print("EQL PERFORMANCE")
print("============================================================")

print("\nTRAIN")

for key, value in train_metrics.items():

    print(
        f"{key:12s}: "
        f"{value:.6f}"
    )


print("\nFROZEN TEST")

for key, value in test_metrics.items():

    print(
        f"{key:12s}: "
        f"{value:.6f}"
    )


# ============================================================
# 17) EXTRACT RAW EQL EQUATION
#
# IMPORTANT:
# reduce=False avoids the very expensive SymPy simplify()
# that caused the previous KeyboardInterrupt.
# ============================================================

print("\n============================================================")
print("LEARNED EQL EQUATION")
print("============================================================")


equation = model.formula(
    raw_latex=False,
    reduce=False
)


print("\nRAW EQL EQUATION:")
print(equation)


# ============================================================
# 18) SAVE RAW EQUATION
# ============================================================

equation_file = os.path.join(
    RESULT_DIR,
    "EQL_Controlled_raw_equation.txt"
)


with open(
    equation_file,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        str(equation)
    )


# ============================================================
# 19) SAVE TEST PREDICTIONS
# ============================================================

test_results = X_test.copy()

test_results["HFT_Actual_K"] = y_test

test_results["HFT_Predicted_K"] = pred_test

test_results["Error_K"] = (
    pred_test - y_test
)

test_results["Abs_Error_K"] = np.abs(
    pred_test - y_test
)

test_results.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_test_predictions.csv"
    ),
    index=False
)


# ============================================================
# 20) SAVE TRAIN PREDICTIONS
# ============================================================

train_results = X_train.copy()

train_results["HFT_Actual_K"] = y_train

train_results["HFT_Predicted_K"] = pred_train

train_results["Error_K"] = (
    pred_train - y_train
)

train_results["Abs_Error_K"] = np.abs(
    pred_train - y_train
)

train_results.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_train_predictions.csv"
    ),
    index=False
)


# ============================================================
# 21) SAVE SCALING PARAMETERS
# ============================================================

scaling_parameters = pd.DataFrame({
    "Variable": ["SG", "P"],
    "Mean": [
        X_mean["SG"],
        X_mean["P"]
    ],
    "Std": [
        X_std["SG"],
        X_std["P"]
    ]
})


scaling_parameters.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_scaling_parameters.csv"
    ),
    index=False
)


# ============================================================
# 22) SAVE METRICS
# ============================================================

metrics_table = pd.DataFrame([
    {
        "Dataset": "Train",
        **train_metrics
    },
    {
        "Dataset": "Frozen_Test",
        **test_metrics
    }
])


metrics_table.to_csv(
    os.path.join(
        RESULT_DIR,
        "EQL_Controlled_metrics.csv"
    ),
    index=False
)


# ============================================================
# 23) FINAL SUMMARY
# ============================================================

print("\n============================================================")
print("FINAL EQL CONTROLLED SUMMARY")
print("============================================================")

print(
    f"Train RRMSE : "
    f"{train_metrics['RRMSE_%']:.6f} %"
)

print(
    f"Test RRMSE  : "
    f"{test_metrics['RRMSE_%']:.6f} %"
)

print(
    f"Test RMSE   : "
    f"{test_metrics['RMSE_K']:.6f} K"
)

print(
    f"Test MAE    : "
    f"{test_metrics['MAE_K']:.6f} K"
)

print(
    f"Test R²     : "
    f"{test_metrics['R2']:.6f}"
)

print("\nResults saved in:")
print(
    os.path.abspath(
        RESULT_DIR
    )
)

print("\nDone.")


FROZEN GROUPED SPLIT
Total observations : 3811
Training           : 3048
Testing            : 763
Training groups    : 3042
Testing groups     : 761
Group overlap      : 0

EQL OPERATOR INFORMATION
(w_initializer, b_initializer, v, lmbda=0, mask=None, exclude=None)

EQL MODEL
Model: "model_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_4 (InputLayer)        [(None, 2)]               0         
                                                                 
 eql_layer_6 (EqlLayer)      (None, 5)                 18        
                                                                 
 eql_layer_7 (EqlLayer)      (None, 5)                 36        
                                                                 
 dense_layer_3 (DenseLayer)  (None, 1)                 6         
                                                                 
Total params: 60
Trainable params: 60
Non-tra

### EQUATION FUNCTION

In [15]:
# ============================================================
# EQL RAW EQUATION — DIRECT NUMPY IMPLEMENTATION
#
# x1 = standardized SG
# x2 = standardized Pressure
#
# No SymPy lambdify
# No simplify
# No parsing
# ============================================================

def EQL_RAW_FUNCTION(x1, x2):

    # --------------------------------------------------------
    # Basic EQL internal terms
    # --------------------------------------------------------

    A = (
        -0.142823562026024 * x1
        -1.24822020530701 * x2
        +1.31131029129028
    )

    B = (
        0.139457061886787 * x1
        -1.28492057323456 * x2
        +1.27211081981659
    )

    AB = A * B


    # --------------------------------------------------------
    # Sigmoid base term
    # --------------------------------------------------------

    sigmoid_base_arg = (
        0.503252685070038 * x1
        +1.1796041727066 * x2
        +2.53373146057129
    )

    sigmoid_base = (
        1.0
        /
        (
            1.0
            +np.exp(
                -sigmoid_base_arg
            )
        )
    )


    # --------------------------------------------------------
    # Sin base term
    # --------------------------------------------------------

    sin_base_arg = (
        -0.0158022623509169 * x1
        +1.00978493690491 * x2
        +1.41976082324982
    )

    sin_base = np.sin(
        sin_base_arg
    )


    # --------------------------------------------------------
    # Cos base term
    # --------------------------------------------------------

    cos_base_arg = (
        1.0592188835144 * x2
        +1.67525839805603
    )

    cos_base = np.cos(
        cos_base_arg
    )


    # --------------------------------------------------------
    # First multiplication branch
    # --------------------------------------------------------

    branch_1 = (
        -1.99055202058631 * x2
        -0.268390953540802 * AB
        -1.99717271327972 * sigmoid_base
        -3.30114483833313 * sin_base
        -1.73285961151123 * cos_base
        -6.81256664737342
    )


    branch_2 = (
        -1.89913792720972 * x2
        -0.194636553525925 * AB
        -2.05707716941833 * sigmoid_base
        -3.3542320728302 * sin_base
        -1.70095539093018 * cos_base
        -6.55121985964024
    )


    # --------------------------------------------------------
    # Nested sigmoid
    # --------------------------------------------------------

    sigmoid_nested_arg = (
        13.4742076878342 * x2
        -4.80180215835571 * AB
        +5.34925508499146 * sigmoid_base
        +31.1683483123779 * sin_base
        +4.63066434860229 * cos_base
        +33.0202969239986
    )

    sigmoid_nested = (
        1.0
        /
        (
            1.0
            +np.exp(
                -sigmoid_nested_arg
            )
        )
    )


    # --------------------------------------------------------
    # Nested sin
    # --------------------------------------------------------

    sin_nested_arg = (
        0.169000894319637 * x2
        -0.340592116117477 * AB
        +1.96202397346497 * sigmoid_base
        +1.98823070526123 * sin_base
        +0.792372345924377 * cos_base
        +0.754542801855266
    )

    sin_nested = np.sin(
        sin_nested_arg
    )


    # --------------------------------------------------------
    # Nested cos
    # --------------------------------------------------------

    cos_nested_arg = (
        1.29767265246898 * x2
        -1.24560511112213 * AB
        +1.90532779693604 * sigmoid_base
        +0.380791246891022 * sin_base
        +6.57446432113647 * cos_base
        +3.44440375786172
    )

    cos_nested = np.cos(
        cos_nested_arg
    )


    # ========================================================
    # FINAL EQL EQUATION
    # ========================================================

    y = (

        7.19034676820822 * x2

        -0.183316466297271
        * AB

        +1.77226448059082
        * branch_1
        * branch_2

        +4.54043722429469
        * sigmoid_base

        +30.1934833526611
        * sigmoid_nested

        +6.82521363053002
        * sin_base

        +19.9305362701416
        * sin_nested

        +1.35163825114523
        * cos_base

        +15.8487777709961
        * cos_nested

        +22.4664211301311
    )

    return y

### EXTERNAL VALIDATION 

In [1]:

# ============================================================
# EQL-SP_GR — EXTERNAL VALIDATION
# DIRECT NUMPY IMPLEMENTATION
#
# IMPORTANT:
#   - No retraining
#   - No new train/test split
#   - No SymPy
#   - No lambdify
#   - No simplify
#   - No equation parsing
#
# Features:
#   x1 = standardized Specific Gravity (SG)
#   x2 = standardized Pressure (psia)
#
# Evaluation:
#   1) All external pressure points
#   2) Pressure < 2500 psia
#
# External datasets:
#   Levan_3, Sample_11, Salman
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# ============================================================
# 1) EQL RAW EQUATION — SP_GR
# DIRECT NUMPY IMPLEMENTATION
# ============================================================

def EQL_RAW_FUNCTION(x1, x2):

    # Basic EQL internal terms
    A = (
        -0.142823562026024 * x1
        -1.24822020530701 * x2
        +1.31131029129028
    )

    B = (
        0.139457061886787 * x1
        -1.28492057323456 * x2
        +1.27211081981659
    )

    AB = A * B

    # Sigmoid base
    sigmoid_base_arg = (
        0.503252685070038 * x1
        +1.1796041727066 * x2
        +2.53373146057129
    )

    sigmoid_base = 1.0 / (
        1.0 + np.exp(-np.clip(sigmoid_base_arg, -700, 700))
    )

    # Sin base
    sin_base_arg = (
        -0.0158022623509169 * x1
        +1.00978493690491 * x2
        +1.41976082324982
    )

    sin_base = np.sin(sin_base_arg)

    # Cos base
    cos_base_arg = (
        1.0592188835144 * x2
        +1.67525839805603
    )

    cos_base = np.cos(cos_base_arg)

    # First nonlinear branch
    branch_1 = (
        -1.99055202058631 * x2
        -0.268390953540802 * AB
        -1.99717271327972 * sigmoid_base
        -3.30114483833313 * sin_base
        -1.73285961151123 * cos_base
        -6.81256664737342
    )

    # Second nonlinear branch
    branch_2 = (
        -1.89913792720972 * x2
        -0.194636553525925 * AB
        -2.05707716941833 * sigmoid_base
        -3.3542320728302 * sin_base
        -1.70095539093018 * cos_base
        -6.55121985964024
    )

    # Nested sigmoid
    sigmoid_nested_arg = (
        13.4742076878342 * x2
        -4.80180215835571 * AB
        +5.34925508499146 * sigmoid_base
        +31.1683483123779 * sin_base
        +4.63066434860229 * cos_base
        +33.0202969239986
    )

    sigmoid_nested = 1.0 / (
        1.0 + np.exp(-np.clip(sigmoid_nested_arg, -700, 700))
    )

    # Nested sin
    sin_nested_arg = (
        0.169000894319637 * x2
        -0.340592116117477 * AB
        +1.96202397346497 * sigmoid_base
        +1.98823070526123 * sin_base
        +0.792372345924377 * cos_base
        +0.754542801855266
    )

    sin_nested = np.sin(sin_nested_arg)

    # Nested cos
    cos_nested_arg = (
        1.29767265246898 * x2
        -1.24560511112213 * AB
        +1.90532779693604 * sigmoid_base
        +0.380791246891022 * sin_base
        +6.57446432113647 * cos_base
        +3.44440375786172
    )

    cos_nested = np.cos(cos_nested_arg)

    # Final EQL equation
    y = (
        7.19034676820822 * x2
        -0.183316466297271 * AB
        +1.77226448059082 * branch_1 * branch_2
        +4.54043722429469 * sigmoid_base
        +30.1934833526611 * sigmoid_nested
        +6.82521363053002 * sin_base
        +19.9305362701416 * sin_nested
        +1.35163825114523 * cos_base
        +15.8487777709961 * cos_nested
        +22.4664211301311
    )

    return y


# ============================================================
# 2) RESULT DIRECTORIES
# ============================================================

RESULT_DIR = "results_EQL_SP_GR_Controlled"
EXTERNAL_RESULT_DIR = os.path.join(
    RESULT_DIR,
    "external_validation"
)

os.makedirs(EXTERNAL_RESULT_DIR, exist_ok=True)


# ============================================================
# 3) ORIGINAL TRAINING-SET SCALING PARAMETERS
# DO NOT RECALCULATE FROM EXTERNAL DATA
# ============================================================

SG_MEAN = 0.771994041840223
SG_STD = 0.14936816650260865

P_MEAN = 1270.7107610901512
P_STD = 1106.6277351650895

if SG_STD <= 0 or P_STD <= 0:
    raise ValueError("Training standard deviations must be positive.")

print("\n============================================================")
print("EQL-SP_GR TRAINING SCALING PARAMETERS")
print("============================================================")
print(f"SG mean : {SG_MEAN:.12f}")
print(f"SG std  : {SG_STD:.12f}")
print(f"P mean  : {P_MEAN:.12f} psia")
print(f"P std   : {P_STD:.12f} psia")


# ============================================================
# 4) EXTERNAL DATASETS
# ============================================================

external_datasets = {

    "Levan_3": {
        "composition": {
            "CH4": 89.19,
            "CO2": 4.44,
            "C2": 4.24,
            "C3": 1.14,
            "iC4": 0.28,
            "nC4": 0.30,
            "iC5": 0.15,
            "nC5": 0.10,
            "C6": 0.09,
            "C7": 0.07
        },

        "P": np.array([
            690, 878.33, 1066.67, 1255, 1443.33,
            1631.67, 1820, 2008.33, 2196.67, 2385,
            2573.33, 2761.67, 2950
        ], dtype=float),

        "HFT": np.array([
            285.90, 287.32, 288.52, 289.64, 290.60,
            291.37, 291.98, 292.48, 292.90, 293.30,
            293.69, 294.07, 294.45
        ], dtype=float)
    },

    "Sample_11": {
        "composition": {
            "CH4": 77.61,
            "CO2": 6.14,
            "C2": 8.81,
            "C3": 3.37,
            "iC4": 0.75,
            "nC4": 1.33,
            "iC5": 0.44,
            "nC5": 0.69,
            "C6": 0.86,
            "C7": 0.00
        },

        "P": np.array([
            590, 702.22, 814.44, 926.67, 1038.89,
            1151.11, 1263.33, 1375.56, 1487.78, 1600
        ], dtype=float),

        "HFT": np.array([
            287.94, 289.30, 290.32, 291.09, 291.66,
            292.12, 292.54, 292.98, 293.53, 294.26
        ], dtype=float)
    },

    "Salman": {
        "composition": {
            "CH4": 74.10,
            "CO2": 3.95,
            "C2": 10.82,
            "C3": 6.70,
            "iC4": 0.94,
            "nC4": 2.04,
            "iC5": 0.53,
            "nC5": 0.58,
            "C6": 0.25,
            "C7": 0.08
        },

        "P": np.array([
            700, 891.67, 1083.33, 1275, 1466.67,
            1658.33, 1850, 2041.67, 2233.33, 2425,
            2616.67, 2808.33, 3000
        ], dtype=float),

        "HFT": np.array([
            289.86, 291.93, 293.02, 293.68, 294.26,
            294.79, 295.26, 295.68, 296.07, 296.44,
            296.79, 297.13, 297.49
        ], dtype=float)
    }
}


# ============================================================
# 5) CALCULATE GAS SPECIFIC GRAVITY FROM COMPOSITION
#
# SG = molecular weight of gas mixture / molecular weight of air
#
# Gas composition is treated as mole percent.
# Mole fractions are normalized before calculating mixture MW.
# ============================================================

MOLECULAR_WEIGHTS = {
    "CH4": 16.043,
    "CO2": 44.010,
    "C2": 30.070,
    "C3": 44.097,
    "iC4": 58.124,
    "nC4": 58.124,
    "iC5": 72.151,
    "nC5": 72.151,
    "C6": 86.178,
    "C7": 100.205
}

MW_AIR = 28.967


def calculate_specific_gravity(composition):

    components = list(MOLECULAR_WEIGHTS.keys())

    missing = [
        component for component in components
        if component not in composition
    ]

    if missing:
        raise ValueError(
            f"Missing composition components: {missing}"
        )

    mole_percent = np.array(
        [composition[c] for c in components],
        dtype=float
    )

    if (
        not np.all(np.isfinite(mole_percent))
        or np.any(mole_percent < 0)
        or mole_percent.sum() <= 0
    ):
        raise ValueError(
            "Composition must contain finite, non-negative values "
            "with a positive total."
        )

    mole_fraction = mole_percent / mole_percent.sum()

    molecular_weights = np.array(
        [MOLECULAR_WEIGHTS[c] for c in components],
        dtype=float
    )

    mixture_mw = np.sum(mole_fraction * molecular_weights)
    specific_gravity = mixture_mw / MW_AIR

    return specific_gravity, mixture_mw


# ============================================================
# 6) METRICS
# ============================================================

def calculate_external_metrics(y_true, y_pred):

    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)

    y_true_valid = y_true[valid]
    y_pred_valid = y_pred[valid]

    if len(y_true_valid) == 0:
        raise RuntimeError(
            "No valid prediction points available."
        )

    rmse = np.sqrt(
        mean_squared_error(y_true_valid, y_pred_valid)
    )

    mae = mean_absolute_error(
        y_true_valid, y_pred_valid
    )

    denominator = np.mean(np.abs(y_true_valid))

    rrmse = (
        100.0 * rmse / denominator
        if denominator > 0
        else np.nan
    )

    # R2 is undefined for a single valid point.
    r2 = (
        r2_score(y_true_valid, y_pred_valid)
        if len(y_true_valid) >= 2
        else np.nan
    )

    error = y_pred_valid - y_true_valid

    return {
        "RMSE_K": rmse,
        "MAE_K": mae,
        "RRMSE_%": rrmse,
        "R2": r2,
        "Mean_Error_K": np.mean(error),
        "Max_Abs_Error_K": np.max(np.abs(error)),
        "N_valid": len(y_true_valid),
        "N_invalid": len(y_true) - len(y_true_valid)
    }


# ============================================================
# 7) EXTERNAL VALIDATION
# ============================================================

all_metrics = []
all_predictions = []
dataset_summary = []

print("\n============================================================")
print("EQL-SP_GR EXTERNAL VALIDATION")
print("============================================================")

for dataset_name, dataset in external_datasets.items():

    print("\n------------------------------------------------------------")
    print(dataset_name)
    print("------------------------------------------------------------")

    P_values = np.asarray(dataset["P"], dtype=float)
    y_true = np.asarray(dataset["HFT"], dtype=float)

    if len(P_values) != len(y_true):
        raise ValueError(
            f"{dataset_name}: Pressure and HFT lengths do not match."
        )

    if not np.all(np.isfinite(P_values)):
        raise ValueError(
            f"{dataset_name}: Pressure contains non-finite values."
        )

    # --------------------------------------------------------
    # Calculate SG from gas composition
    # --------------------------------------------------------

    SG_value, mixture_mw = calculate_specific_gravity(
        dataset["composition"]
    )

    print(f"Mixture molecular weight : {mixture_mw:.6f}")
    print(f"Calculated SG             : {SG_value:.8f}")

    # --------------------------------------------------------
    # Standardize using ORIGINAL training-set statistics
    # --------------------------------------------------------

    SG_scaled = (SG_value - SG_MEAN) / SG_STD
    P_scaled = (P_values - P_MEAN) / P_STD

    SG_array = np.full(
        len(P_values), SG_scaled, dtype=float
    )

    # --------------------------------------------------------
    # Predict HFT
    # --------------------------------------------------------

    try:
        y_pred = EQL_RAW_FUNCTION(
            SG_array,
            P_scaled
        )
    except Exception as error:
        raise RuntimeError(
            f"Prediction failed for {dataset_name}: {error}"
        ) from error

    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    if len(y_pred) != len(y_true):
        raise ValueError(
            f"{dataset_name}: Prediction length does not match HFT."
        )

    # --------------------------------------------------------
    # Calculate metrics for all pressures and P < 2500 psia
    # --------------------------------------------------------

    pressure_masks = {
        "All_Data": np.ones(len(P_values), dtype=bool),
        "P_below_2500_psia": P_values < 2500.0
    }

    dataset_metric_results = {}

    for pressure_scope, pressure_mask in pressure_masks.items():

        if not np.any(pressure_mask):
            print(
                f"WARNING: {dataset_name} has no points "
                f"for {pressure_scope}."
            )
            continue

        scope_metrics = calculate_external_metrics(
            y_true[pressure_mask],
            y_pred[pressure_mask]
        )

        scope_metrics["Sample"] = dataset_name
        scope_metrics["Pressure_Scope"] = pressure_scope
        scope_metrics["Pressure_Cutoff_psia"] = (
            "All pressures"
            if pressure_scope == "All_Data"
            else "P < 2500 psia"
        )

        scope_metrics["Pressure_Min_psia"] = float(
            np.min(P_values[pressure_mask])
        )

        scope_metrics["Pressure_Max_psia"] = float(
            np.max(P_values[pressure_mask])
        )

        dataset_metric_results[pressure_scope] = scope_metrics
        all_metrics.append(scope_metrics)

    # --------------------------------------------------------
    # Point-by-point errors
    # --------------------------------------------------------

    error = y_pred - y_true
    abs_error = np.abs(error)

    prediction_table = pd.DataFrame({
        "Sample": dataset_name,
        "Calculated_SG": SG_value,
        "Mixture_MW": mixture_mw,
        "Pressure_psia": P_values,
        "Included_in_P_below_2500": P_values < 2500.0,
        "HFT_Actual_K": y_true,
        "HFT_Predicted_K": y_pred,
        "Error_K": error,
        "Abs_Error_K": abs_error
    })

    all_predictions.append(prediction_table)

    # --------------------------------------------------------
    # Dataset summary
    # --------------------------------------------------------

    dataset_summary.append({
        "Sample": dataset_name,
        "Calculated_SG": SG_value,
        "Mixture_MW": mixture_mw,
        "N_points": len(P_values),
        "N_points_below_2500": int(
            np.sum(P_values < 2500.0)
        ),
        "Pressure_Min_psia": np.min(P_values),
        "Pressure_Max_psia": np.max(P_values),
        "HFT_Min_K": np.min(y_true),
        "HFT_Max_K": np.max(y_true)
    })

    # --------------------------------------------------------
    # Console output: metrics for both scopes
    # --------------------------------------------------------

    for scope in ["All_Data", "P_below_2500_psia"]:

        if scope not in dataset_metric_results:
            continue

        metrics = dataset_metric_results[scope]

        print(f"\nPressure scope: {scope}")
        print(f"RMSE        : {metrics['RMSE_K']:.6f} K")
        print(f"RRMSE       : {metrics['RRMSE_%']:.6f} %")
        print(f"MAE         : {metrics['MAE_K']:.6f} K")
        print(f"R2          : {metrics['R2']:.6f}")
        print(f"Mean Error  : {metrics['Mean_Error_K']:.6f} K")
        print(f"Max Abs Err : {metrics['Max_Abs_Error_K']:.6f} K")
        print(f"N valid     : {metrics['N_valid']}")


# ============================================================
# 8) COMBINE RESULTS
# ============================================================

metrics_df = pd.DataFrame(all_metrics)
predictions_df = pd.concat(
    all_predictions,
    ignore_index=True
)
summary_df = pd.DataFrame(dataset_summary)


# ============================================================
# 9) SAVE METRICS
# ============================================================

metrics_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_SP_GR_External_Metrics.csv"
)

metrics_df.to_csv(metrics_file, index=False)


# ============================================================
# 10) SIDE-BY-SIDE METRICS COMPARISON
# ============================================================

comparison_metric_columns = [
    "RMSE_K",
    "MAE_K",
    "RRMSE_%",
    "R2",
    "Mean_Error_K",
    "Max_Abs_Error_K",
    "N_valid",
    "N_invalid"
]

comparison_rows = []

for sample_name in external_datasets:

    sample_rows = metrics_df[
        metrics_df["Sample"] == sample_name
    ]

    row = {"Sample": sample_name}

    for scope, suffix in [
        ("All_Data", "All_Data"),
        ("P_below_2500_psia", "P_below_2500_psia")
    ]:

        scope_row = sample_rows[
            sample_rows["Pressure_Scope"] == scope
        ]

        for metric_name in comparison_metric_columns:

            output_name = f"{metric_name}_{suffix}"

            row[output_name] = (
                scope_row.iloc[0][metric_name]
                if not scope_row.empty
                else np.nan
            )

    # Filtered metric minus full-dataset metric
    for metric_name in [
        "RMSE_K",
        "MAE_K",
        "RRMSE_%",
        "R2"
    ]:

        all_value = row.get(
            f"{metric_name}_All_Data", np.nan
        )

        filtered_value = row.get(
            f"{metric_name}_P_below_2500_psia", np.nan
        )

        row[f"Delta_{metric_name}_Filtered_minus_All"] = (
            filtered_value - all_value
        )

    comparison_rows.append(row)

metrics_comparison_df = pd.DataFrame(comparison_rows)

comparison_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_SP_GR_External_Metrics_Comparison.csv"
)

metrics_comparison_df.to_csv(
    comparison_file,
    index=False
)


# ============================================================
# 11) SAVE POINT-BY-POINT PREDICTIONS
# ============================================================

predictions_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_SP_GR_External_Predictions.csv"
)

predictions_df.to_csv(
    predictions_file,
    index=False
)


# ============================================================
# 12) SAVE DATASET SUMMARY
# ============================================================

summary_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_SP_GR_External_Dataset_Summary.csv"
)

summary_df.to_csv(
    summary_file,
    index=False
)


# ============================================================
# 13) SAVE TRAINING SCALING PARAMETERS
# ============================================================

scaling_df = pd.DataFrame({
    "Variable": ["SG", "P"],
    "Training_Mean": [SG_MEAN, P_MEAN],
    "Training_Std": [SG_STD, P_STD]
})

scaling_file = os.path.join(
    EXTERNAL_RESULT_DIR,
    "EQL_SP_GR_External_Scaling_Parameters.csv"
)

scaling_df.to_csv(
    scaling_file,
    index=False
)


# ============================================================
# 14) PRINT FINAL METRICS TABLE
# ============================================================

print("\n============================================================")
print("FINAL EQL-SP_GR EXTERNAL VALIDATION METRICS")
print("============================================================")

display_columns = [
    "Sample",
    "Pressure_Scope",
    "Pressure_Min_psia",
    "Pressure_Max_psia",
    "RMSE_K",
    "RRMSE_%",
    "MAE_K",
    "R2",
    "Mean_Error_K",
    "Max_Abs_Error_K",
    "N_valid",
    "N_invalid"
]

print(
    metrics_df[display_columns].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)


# ============================================================
# 15) PRINT SIDE-BY-SIDE COMPARISON
# ============================================================

print("\n============================================================")
print("METRICS COMPARISON: ALL DATA vs P < 2500 psia")
print("============================================================")

print(
    metrics_comparison_df.to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)


# ============================================================
# 16) PRINT POINT-BY-POINT PREDICTIONS
# ============================================================

print("\n============================================================")
print("POINT-BY-POINT EXTERNAL PREDICTIONS")
print("============================================================")

print(
    predictions_df[
        [
            "Sample",
            "Calculated_SG",
            "Pressure_psia",
            "Included_in_P_below_2500",
            "HFT_Actual_K",
            "HFT_Predicted_K",
            "Error_K",
            "Abs_Error_K"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.4f}"
    )
)


# ============================================================
# 17) FILES SAVED
# ============================================================

print("\n============================================================")
print("FILES SAVED")
print("============================================================")

print(os.path.abspath(EXTERNAL_RESULT_DIR))
print("\n1. EQL_SP_GR_External_Metrics.csv")
print("2. EQL_SP_GR_External_Metrics_Comparison.csv")
print("3. EQL_SP_GR_External_Predictions.csv")
print("4. EQL_SP_GR_External_Dataset_Summary.csv")
print("5. EQL_SP_GR_External_Scaling_Parameters.csv")

print("\n============================================================")
print("EQL-SP_GR EXTERNAL VALIDATION COMPLETED")
print("============================================================")



EQL-SP_GR TRAINING SCALING PARAMETERS
SG mean : 0.771994041840
SG std  : 0.149368166503
P mean  : 1270.710761090151 psia
P std   : 1106.627735165089 psia

EQL-SP_GR EXTERNAL VALIDATION

------------------------------------------------------------
Levan_3
------------------------------------------------------------
Mixture molecular weight : 18.705670
Calculated SG             : 0.64575793

Pressure scope: All_Data
RMSE        : 0.435697 K
RRMSE       : 0.149597 %
MAE         : 0.358402 K
R2          : 0.971850
Mean Error  : 0.260702 K
Max Abs Err : 0.845234 K
N valid     : 13

Pressure scope: P_below_2500_psia
RMSE        : 0.460486 K
RRMSE       : 0.158569 %
MAE         : 0.395356 K
R2          : 0.962344
Mean Error  : 0.280374 K
Max Abs Err : 0.845234 K
N valid     : 10

------------------------------------------------------------
Sample_11
------------------------------------------------------------
Mixture molecular weight : 22.053839
Calculated SG             : 0.76134355

Pressu